# Importing packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

Load data from pre-downloaded Oracle Elixer's Data from 2014 to 2025 to a local persistent database file for quick access on future sessions. This is for testing simple models.

In [2]:
# Connect to persistent database file on disk
# con = duckdb.connect("lol_predictors.duckdb")

# oe_data_path = Path("./OE datasets")

# for dataset in oe_data_path.iterdir():
#     if dataset.is_file() and dataset.suffix == ".csv":
#         var_name = f"df_{dataset.name[:4]}" # e.g., df_2014
        
#         # Read CSV via pandas
#         df = pd.read_csv(dataset, low_memory=False)
        
#         # Save directly into a permanent DuckDB table in your file!
#         con.execute(f"CREATE OR REPLACE TABLE {var_name} AS SELECT * FROM df")
#         print(f"Saved {var_name} into DuckDB file.")

# con.close()

# Naive Baseline Model
Using the winrate of the previous split, the baseline model predicts the results of the next split.

In [3]:
# Connect to saved database file
con = duckdb.connect("lol_predictors.duckdb")

In [4]:
baseline_query_2014 = con.execute("""
    SELECT 
        teamname, 
        CAST(SUM(result) AS DOUBLE) / COUNT(*) as win_rate,
        COUNT(*) as games_played
    FROM df_2014
    WHERE position = 'team' AND split = 'Spring'
    GROUP BY teamname
""").df()

In [5]:
test_matches = con.execute("""
    SELECT 
        teamname,
        result
    FROM df_2014
    WHERE position = 'team' AND split = 'Summer'
""").df()

In [6]:
con.close()

In [7]:
merged = test_matches.merge(baseline_query_2014, on='teamname', how='left')
merged['win_rate'] = merged['win_rate'].fillna(0.5)

In [8]:
merged['prediction'] = (merged['win_rate'] >= 0.5).astype(int)

In [9]:
from sklearn.metrics import accuracy_score

In [10]:
acc = accuracy_score(merged['result'], merged['prediction'])
print(f"Baseline Accuracy: {acc * 100:.2f}%")

Baseline Accuracy: 53.27%


# Logistic Regression Model

In [11]:
from sklearn.linear_model import LogisticRegression